In [ ]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from autoencoder_anomaly_detector import (
    ThresholdConfig,
    TrainConfig,
    VAEConfig,
    binary_metrics,
    classify,
    find_latest_processed_window_run,
    load_window_run,
    mse_reconstruction_scores,
    roc_auc_binary,
    save_training_artifacts,
    select_threshold,
    train_vae,
)

try:
    from sklearn.ensemble import IsolationForest
except ImportError as exc:
    raise ImportError('scikit-learn is required for Isolation Forest. Install with: pip install scikit-learn') from exc

In [ ]:
# Choose model type to train
MODEL_TYPE = 'isolation_forest'  # 'autoencoder' or 'isolation_forest'
if MODEL_TYPE not in ['autoencoder', 'isolation_forest']:
    raise ValueError(f'Invalid MODEL_TYPE: {MODEL_TYPE}. Must be "autoencoder" or "isolation_forest".')

MODEL_OUTPUT_ROOT = PROJECT_ROOT / 'models' / 'autoencoder'
if MODEL_TYPE == 'isolation_forest':
    MODEL_OUTPUT_ROOT = PROJECT_ROOT / 'models' / 'isolation_forest'
PROCESSED_WINDOWS_ROOT = PROJECT_ROOT / 'dataset' / 'processed_windows'

# Toggle this to skip loading an external validation split.
USE_EXTERNAL_VALIDATION_SPLIT = True

ENFORCE_TRAIN_NORMAL_ONLY = True

selected_source_run = find_latest_processed_window_run(PROCESSED_WINDOWS_ROOT, require_val=USE_EXTERNAL_VALIDATION_SPLIT)

train_windows, val_windows, _, train_labels, val_labels, _, metadata = load_window_run(selected_source_run)
if not USE_EXTERNAL_VALIDATION_SPLIT or len(val_windows) == 0:
    val_windows = None
    val_labels = np.array([], dtype=np.int32)

if train_windows.ndim != 3:
    raise ValueError('Expected windowed training data with shape (n_windows, window_size, n_features).')

print('Loaded window arrays from processed_windows:')
print(f' - run directory: {selected_source_run}')
print(f' - train_windows: {train_windows.shape}')
if USE_EXTERNAL_VALIDATION_SPLIT and val_windows is not None:
    print(f' - val_windows:   {val_windows.shape}')
else:
    print(' - validation split: disabled')

if ENFORCE_TRAIN_NORMAL_ONLY and np.any(train_labels == 1):
    raise ValueError('Training data contains anomaly labels (1). Training must be normal-only.')

print(f' - train_labels positives: {int(train_labels.sum())}')
if USE_EXTERNAL_VALIDATION_SPLIT and len(val_labels):
    print(f' - val_labels positives:   {int(val_labels.sum())}')

In [ ]:
if MODEL_TYPE == 'autoencoder':
    # Core architecture params
    vae_cfg = VAEConfig(
        architecture='dense'  # 'dense', 'conv1d', or 'lstm_autoencoder'
    )

    # Optimization and loss weighting
    train_cfg = TrainConfig(
        beta=0,  # beta=0 + True => deterministic AE; beta=0 + False => stochastic AE; beta=1 => VAE
        sample_from_mu=True,
    )

    vae_cfg, train_cfg
elif MODEL_TYPE == 'isolation_forest':
    iso_cfg = {
        'n_estimators': 300,
        'max_samples': 'auto',
        'contamination': 'auto',
        'random_state': 42,
        'n_jobs': -1,
    }
    iso_cfg

In [ ]:
if MODEL_TYPE == 'autoencoder':
    encoder, decoder, history = train_vae(
        train_windows,
        vae_cfg,
        train_cfg,
        val_windows=val_windows,
    )

    history_df = pd.DataFrame(history)
    print('Epoch-by-epoch loss report:')
    if USE_EXTERNAL_VALIDATION_SPLIT:
        display(history_df)
        loss_cols = ['train_total_loss', 'val_total_loss']
        loss_label = 'train/val total loss'
    else:
        display(history_df[['epoch', 'train_total_loss', 'train_recon_loss', 'train_kl_loss']])
        loss_cols = ['train_total_loss']
        loss_label = 'train total loss'

    non_finite = ~np.isfinite(history_df[loss_cols].to_numpy(dtype=np.float64))
    print(f'Any non-finite {loss_label}: {bool(non_finite.any())}')
elif MODEL_TYPE == 'isolation_forest':
    # Flatten windows for Isolation Forest (n_windows, window_size * n_features).
    train_flat = train_windows.reshape(len(train_windows), -1)
    val_flat = (
        val_windows.reshape(len(val_windows), -1)
        if val_windows is not None and len(val_windows)
        else None
    )

    iso = IsolationForest(**iso_cfg)
    iso.fit(train_flat)
    print('Isolation Forest training complete.')

In [ ]:
# Thresholding strategy for anomaly decision
threshold_cfg = ThresholdConfig(
    method='val_f1',  # 'val_f1' to select threshold based on validation F1 score; 'p-train' to select based on train scores distribution
)

if MODEL_TYPE == 'autoencoder':
    # Reconstruction MSE scores (only needed for threshold selection during training)
    train_scores = mse_reconstruction_scores(encoder, decoder, train_windows, batch_size=train_cfg.batch_size)
    val_scores = (
        mse_reconstruction_scores(encoder, decoder, val_windows, batch_size=train_cfg.batch_size)
        if val_windows is not None and len(val_windows)
        else np.array([], dtype=np.float32)
    )
    have_external_validation = (
        USE_EXTERNAL_VALIDATION_SPLIT
        and val_windows is not None
        and len(val_scores)
        and np.unique(val_labels).size >= 2
    )
elif MODEL_TYPE == 'isolation_forest':
    # Isolation Forest score_samples: higher = more normal. Flip to make higher = more anomalous.
    train_scores = -iso.score_samples(train_flat)
    val_scores = (
        -iso.score_samples(val_flat)
        if val_flat is not None and len(val_flat)
        else np.array([], dtype=np.float32)
    )
    have_external_validation = (
        USE_EXTERNAL_VALIDATION_SPLIT
        and val_flat is not None
        and len(val_scores)
        and np.unique(val_labels).size >= 2
    )

# Select threshold based on validation data when it exists; otherwise fall back to train-only thresholding.
if threshold_cfg.method == 'val_f1' and have_external_validation:
    threshold, threshold_info = select_threshold(
        train_scores=train_scores,
        cfg=threshold_cfg,
        val_scores=val_scores,
        val_labels=val_labels,
    )
else:
    fallback_cfg = ThresholdConfig(
        method='percentile',
        percentile=threshold_cfg.percentile,
        std_factor=threshold_cfg.std_factor,
    )
    threshold, threshold_info = select_threshold(train_scores=train_scores, cfg=fallback_cfg)
    if threshold_cfg.method == 'val_f1' and USE_EXTERNAL_VALIDATION_SPLIT:
        print('val_f1 fallback -> percentile threshold (validation labels are not suitable).')

# Compute training metrics
train_preds = classify(train_scores, threshold)
train_metrics = binary_metrics(train_labels, train_preds)

# Compute validation metrics with AUC if available
if have_external_validation:
    val_preds = classify(val_scores, threshold)
    val_metrics = binary_metrics(val_labels, val_preds)
    val_metrics['roc_auc'] = roc_auc_binary(val_labels, val_scores)
    print('\nValidation Metrics:')
    display(pd.DataFrame([val_metrics]))

print(f'\nThreshold: {threshold:.6f}')
print(f'Threshold info: {threshold_info}')

In [ ]:
if MODEL_TYPE == 'autoencoder':
    saved_run_dir, summary = save_training_artifacts(
        output_root=MODEL_OUTPUT_ROOT,
        encoder=encoder,
        decoder=decoder,
        history=history,
        train_scores=train_scores,
        threshold=threshold,
        train_preds=train_preds,
        train_metrics=train_metrics,
        vae_cfg=vae_cfg,
        train_cfg=train_cfg,
        threshold_cfg=threshold_cfg,
        source_run_dir=selected_source_run,
        project_root=PROJECT_ROOT,
    )

    print(f'Model artifacts saved to: {saved_run_dir}')
    print('Training completed. Use evaluate_models.ipynb to load and test this model.')
    print(f'\nTraining Summary:')
    print(f'  - Threshold: {threshold:.6f}')
    print(f'  - Train Accuracy: {train_metrics["accuracy"]:.6f}')
    display(pd.json_normalize(summary, sep='.').T.rename(columns={0: 'value'}))
elif MODEL_TYPE == 'isolation_forest':
    def _to_relative(path_value: str | Path) -> str:
        path_obj = Path(path_value)
        if not path_obj.is_absolute():
            return path_obj.as_posix()
        try:
            return path_obj.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
        except Exception:
            return str(path_value)

    run_id = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    run_dir = MODEL_OUTPUT_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    model_path = run_dir / 'isolation_forest.pkl'
    with model_path.open('wb') as f:
        pickle.dump(iso, f)

    np.save(run_dir / 'train_scores.npy', train_scores)
    np.save(run_dir / 'train_predictions.npy', train_preds)

    summary = {
        'run_id': run_id,
        'source_data_run': _to_relative(selected_source_run),
        'model_type': 'isolation_forest',
        'model_config': iso_cfg,
        'threshold_config': {
            'method': threshold_cfg.method,
            'percentile': threshold_cfg.percentile,
            'std_factor': threshold_cfg.std_factor,
        },
        'threshold': float(threshold),
        'train_samples': int(len(train_scores)),
        'train_metrics': train_metrics,
        'saved_files': {
            'model': _to_relative(model_path),
            'train_scores': _to_relative(run_dir / 'train_scores.npy'),
            'train_predictions': _to_relative(run_dir / 'train_predictions.npy'),
        },
    }

    summary_path = run_dir / 'summary.json'
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print(f'Model artifacts saved to: {run_dir}')
    print('Training completed. Use evaluate_models.ipynb to load and test this baseline run.')